***PROYECTO SQL***

El objetivo es generar una propuesta de valor para un nuevo producto basado en los servicios que compiten en el mercado de los libros. El DataFrame contiene editoriales, autores y calificaciones de clientes, reseñas de libros. 

In [4]:
#Librerías 

import pandas as pd
from sqlalchemy import create_engine

In [21]:
#Conectar con la base de datos 
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-final-project-db'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

# Crear la conexión
engine = create_engine(connection_string, connect_args={'sslmode':'require'})

# leer la tabla 'books'
print('------- BOOKS -----')
query_test = "SELECT * FROM books LIMIT 5"
df_test = pd.io.sql.read_sql(query_test, con=engine)
print(df_test)
#leer la tabla 'authors'
print()
print('------- AUTHORS -----')
query_test2 = "SELECT * FROM authors LIMIT 5"
df_test2 = pd.io.sql.read_sql(query_test2, con=engine)
print(df_test2)
#leer la tabla 'ratings'
print()
print('------- RATINGS -----')
query_test3 = "SELECT * FROM ratings LIMIT 5"
df_test3 = pd.io.sql.read_sql(query_test3, con=engine)
print(df_test3)
#Leer la tabla 'reviews'
print()
print('------- REVIEWS -----')
query_test4 = "SELECT * FROM reviews LIMIT 5"
df_test4 = pd.io.sql.read_sql(query_test4, con=engine)
print(df_test4)
#Leer la tabla 'publishers'
print()
print('------- PUBLISHERS -----')
query_test5 = "SELECT * FROM publishers LIMIT 5"
df_test5 = pd.io.sql.read_sql(query_test5, con=engine)
print(df_test5)
#

------- BOOKS -----
   book_id  author_id                                              title  \
0        1        546                                       'Salem's Lot   
1        2        465                 1 000 Places to See Before You Die   
2        3        407  13 Little Blue Envelopes (Little Blue Envelope...   
3        4         82  1491: New Revelations of the Americas Before C...   
4        5        125                                               1776   

   num_pages publication_date  publisher_id  
0        594       2005-11-01            93  
1        992       2003-05-22           336  
2        322       2010-12-21           135  
3        541       2006-10-10           309  
4        386       2006-07-04           268  

------- AUTHORS -----
   author_id                          author
0          1                      A.S. Byatt
1          2  Aesop/Laura Harris/Laura Gibbs
2          3                 Agatha Christie
3          4                   Alan Brennert

In [22]:
#Número de libros publicados después del 1 de enero de 2000
query_1 = """
SELECT
    COUNT(book_id)
FROM
    books
WHERE
    publication_date > '2000-01-01'
"""


result_1 = pd.io.sql.read_sql(query_1, con=engine)
print(result_1)


   count
0    819


In [13]:
#Número de reseñas de usuario y la calificación promedio de cada libro 
query_2 = """
SELECT
    b.title,
    review_counts.total_reviews,
    avg_ratings.avg_rating
FROM
    books b
LEFT JOIN (
    SELECT book_id, COUNT(review_id) AS total_reviews
    FROM
    reviews 
    GROUP BY book_id
)AS review_counts ON b.book_id = review_counts.book_id
LEFT JOIN (
    SELECT book_id, AVG(rating) AS avg_rating
    FROM ratings
    GROUP BY book_id
)AS avg_ratings ON b.book_id = avg_ratings.book_id;
"""

#la tabla books se apoda como b

In [12]:
result_2 = pd.io.sql.read_sql(query_2, con=engine)
print(result_2.head())

                                               title  total_reviews  \
0          The Body in the Library (Miss Marple  #3)            2.0   
1                                          Galápagos            2.0   
2                           A Tree Grows in Brooklyn            5.0   
3  Undaunted Courage: The Pioneering First Missio...            2.0   
4                                        The Prophet            4.0   

   avg_rating  
0    4.500000  
1    4.500000  
2    4.250000  
3    4.000000  
4    4.285714  


In [14]:
#Libros con más de 50 páginas 
query_3 = """
SELECT 
    p.publisher,
    COUNT(b.book_id) AS total_books
FROM
   publishers p 
INNER JOIN
    books b ON p.publisher_id = b.publisher_id
WHERE 
    b.num_pages > 50
GROUP BY
    p.publisher
ORDER BY
    total_books DESC
LIMIT 1;
"""

In [15]:
result_3 = pd.io.sql.read_sql(query_3, con=engine)
print(result_3)

       publisher  total_books
0  Penguin Books           42


In [16]:
#Calificación promedio más alta (libros con al menos 50 calificaciones)
query_4="""
SELECT
    a.author,
    AVG(r.rating) AS avg_rating
FROM
    authors a
JOIN 
    books b ON a.author_id = b.author_id
JOIN
    ratings r ON b.book_id = r.book_id
WHERE
    b.book_id IN(
        SELECT book_id
        FROM ratings
        GROUP BY book_id
        HAVING COUNT(rating)>=50
        
    )
    GROUP BY
        a.author
    ORDER BY
        avg_rating DESC
    LIMIT 1;
    """

In [17]:

result_4 = pd.io.sql.read_sql(query_4, con=engine)
print(result_4)


                       author  avg_rating
0  J.K. Rowling/Mary GrandPré    4.287097


In [18]:
#Número promedio de reseñas de texto entre los usuarios que calificaron +50 libros 
query_5 ="""
SELECT 
    AVG(review_count) AS avg_reviews_per_expert
FROM ( 
    SELECT
        username,
        COUNT(review_id) AS review_count
    FROM
        reviews
    WHERE 
        username IN(
        SELECT username
        FROM ratings
        GROUP BY username
        HAVING COUNT(rating)>50
        )
    GROUP BY
        username
)AS subquery;
    
"""


In [19]:
result_5 = pd.io.sql.read_sql(query_5, con=engine)
print(result_5)

   avg_reviews_per_expert
0               24.333333


- Hay 819 libros nuevos post 2000, por ende es una base de datos actualizada con contenido moderno
- Se identificaron los 5 libros que generan más conversación
- Penguin Books con 42 libros es el proveedor más importante de contenido de calidad
- J.K Rowling es la escritora que atrae masas
- Los usuarios más activos (con más de 50 calificaciones) mantienen un promedio de 24.33 reseñas escritas. Esto indica que los usuarios más leales son también los principales creadores de contenido, lo que sugiere que la nueva aplicación debería incluir funciones que faciliten la escritura de reseñas largas o foros de discusión.